# Create local networks from pbf

This notebook loads a pbf file of whole europe via pyrosm, then uses city boundary geojson files of all European cities (above 100k population) to extract and export local street and bike networks as gpkg files.

Preliminary step: All .osm.pbf country files need to be downloaded from [geofabrik](https://download.geofabrik.de/europe.html), renamed to just `countryname.osm.pbf`, and placed in the countries folder.

# Load packages

In [ ]:
import pyrosm
pyrosm.__version__
import csv
from growbikenet.functions import *
from growbikenet import constants
from growbikenet import settings
from slugify import slugify

## Parameters

In [ ]:
countries_path = "../countries/"
boundaries_folder = "boundaries/"
cities_path = "../cities/"
cityfilename = "european_capitalsand100000pop.csv"
output_folder = "cityexport/"
dropnodedata = ['osmid','tags','version','changeset','visible'] # osmid is necessary to drop, as will be reinserted
dropedgedata = ['bicycle','busway','cycleway','est_width','foot','int_ref','lit','motor_vehicle','oneway:bicycle','passing_places','sidewalk','smoothness','surface','tracktype','width','timestamp','version','tags','osm_type']

## Load Europe and cities

In [ ]:
with open(cities_folder+cityfilename, mode='r') as infile:
    reader = csv.reader(infile, delimiter=";")
    header = next(reader)
    cities = {slugify(rows[0]): {header[0]: rows[0], header[1]: rows[1], header[2]: rows[2], header[3]: rows[3], "boundaryfile": slugify(rows[0])+"_"+slugify(rows[3])} for rows in reader}

## Run

In [ ]:
for cityid, city_info in cities.items():
    print(city_info["name_en"]+", "+city_info["country_en"])
    # Load city boundary
    try:
        boundary = gpd.read_file(cities_path+boundaries_folder+city_info["boundaryfile"]+".geojson")
    except:
        boundary = gpd.read_file(cities_path+boundaries_folder+city_info["boundaryfile"]+".shp")

    # Create city osm object from country pbf
    countrypbf_file = countries_path+slugify(city_info["country_en"])+".osm.pbf"
    citypbf_file = cities_path+output_folder+city_info["boundaryfile"]+".osm.pbf"
    citygpkg_file = cities_path+output_folder+city_info["boundaryfile"]+".gpkg"
    if not os.path.exists(citypbf_file): # Crop country osm object to envelope of city boundary, save as pbf, and re-load as city osm object
        osm_country = pyrosm.OSM(countrypbf_file, bounding_box=boundary.loc[0, 'geometry'])
        osm_country.to_pbf(output_path=citypbf_file)
    osm = pyrosm.OSM(citypbf_file)

    # Extract street network from pyrosm's city osm object, truncate to boundary, and save as city gpkg
    if not os.path.exists(citygpkg_file): # Create and save
        (street_nodes, street_edges) = osm.get_network(network_type="driving", nodes=True)
        street_network = pyrosm.OSM.to_graph(street_nodes, street_edges, graph_type='networkx', direction='oneway', from_id_col='u', to_id_col='v', edge_id_col='id', node_id_col='id', force_bidirectional=True, retain_all=False, osmnx_compatible=True, simplify=False)
        street_network = ox.truncate.truncate_graph_polygon(street_network, boundary.to_crs("4326").loc[0, 'geometry'])
        ebunch = set()
        for u, v, data in street_network.edges(data=True):
            if data["highway"] == "service": # for some reason pyrosm did not remove service roads although driving was selected (and not driving+service)
                ebunch.add((u,v))
        street_network.remove_edges_from(ebunch)
        street_network = ox.simplification.simplify_graph(street_network)
        street_network = ox.truncate.largest_component(street_network)
        street_network.remove_nodes_from(list(nx.isolates(street_network)))
    
        for u, data in street_network.nodes(data=True):
            for k in dropnodedata:
                del data[k]
        for u, v, data in street_network.edges(data=True):
            for k in dropedgedata:
                del data[k]
                
        ox.io.save_graph_geopackage(street_network, citygpkg_file)
    else: # Load
        street_edges = gpd.read_file(citygpkg_file, layer='edges')
        street_nodes = gpd.read_file(citygpkg_file, layer='nodes')
        street_network = ox.convert.graph_from_gdfs(street_nodes, street_edges)
    

    # Extract bike network from pyrosm's osm object

    # Save as gpkg with 3 layers boundary, street network, bike network
    